# 🎓 Tutorial 2: Building Features via YAML Config

Now that we have messy data, we need to clean it and build "Features" (like a rolling 24-hour max heart rate).

In a standard pipeline, you might write a massive Python file with 50 different `groupby().rolling().max()` commands. In our pipeline, we use **Config-Driven Engineering**.

## Step 1: Look at the Feature Configuration

Let's look at `feature_config.yaml`. Notice how we just *declare* what we want to happen, and the pipeline does it automatically.

In [4]:
import yaml
from pathlib import Path
import sys

project_root = Path().resolve().parent
sys.path.append(str(project_root))

config_path = project_root / 'config' / 'feature_config.yaml'
with open(config_path, 'r') as f:
    feature_config = yaml.safe_load(f)

print("How we process Heart Rate (hr):\n")
print(yaml.dump(feature_config['base_features']['hr']))

How we process Heart Rate (hr):

delta: true
impute: ffill
missing_indicator: false
rolling:
  aggs:
  - mean
  - max
  windows:
  - 24h
  - 7D



## Step 2: Load the Messy Data

We will load the latest synthetic data we generated.

In [5]:
import pandas as pd
from src.utils import get_latest_data_dir

latest_data_dir = get_latest_data_dir()
df_static = pd.read_csv(latest_data_dir / 'df_static.csv')
df_ts_ms = pd.read_csv(latest_data_dir / 'df_ts_missing.csv', parse_dates=['date'])

print("Raw Time-Series Data:")
display(df_ts_ms[['patient_id', 'date', 'hr', 'sbp']].head(3))

Raw Time-Series Data:


,patient_id,date,hr,sbp
0,101,2024-01-01 00:00:00,52.04,51.92
1,101,2024-01-01 04:00:00,60.63,53.69
2,101,2024-01-01 08:00:00,62.13,NaN


## Step 3: Run the Feature Pipeline

Watch what happens when we pass this raw data into the `FeaturePipeline`. It reads the YAML file, fills in the missing values (imputation), and generates all our rolling windows and pandas expressions!

In [7]:
from src.features import FeaturePipeline

# Initialize the pipeline with our YAML config
pipeline = FeaturePipeline(config_path=config_path)

# Process the data!
final_features = pipeline.process(df_static, df_ts_ms)

print("\n✅ Feature Engineering Complete!")
print("Notice how we now have 'hr_24h_mean' and 'hr_24h_max' columns!")
display(final_features[['patient_id', 'date', 'hr', 'hr_24h_mean', 'hr_24h_max', 'qsofa_score']].head(10))

Processing base features...
Computing pandas eval expressions...
Executing custom Python scores...
⚠️ Warning: [Palacios-Baena 2019] Missing critical variables: ['bsi_not_urinary', 'is_non_ecoli']. Results will be underestimated for these records.
⚠️ Warning: Failed to compute custom score 'increment_esbl_score'. Error: 'bsi_not_urinary'

✅ Feature Engineering Complete!
Notice how we now have 'hr_24h_mean' and 'hr_24h_max' columns!


,patient_id,date,hr,hr_24h_mean,hr_24h_max,qsofa_score
0,101,2024-01-01 00:00:00,52.04,52.040000,52.04,True
1,101,2024-01-01 04:00:00,60.63,56.335000,60.63,True
2,101,2024-01-01 08:00:00,62.13,58.266667,62.13,True
3,101,2024-01-01 12:00:00,50.93,56.432500,62.13,True
4,101,2024-01-01 16:00:00,50.93,55.332000,62.13,True
5,101,2024-01-01 20:00:00,40.17,52.805000,62.13,True
6,101,2024-01-02 00:00:00,40.17,50.826667,62.13,True
7,101,2024-01-02 04:00:00,59.91,50.706667,62.13,True
8,101,2024-01-02 08:00:00,59.37,50.246667,59.91,True
9,101,2024-01-02 12:00:00,45.03,49.263333,59.91,True


## Automating the Pipeline

Just like the data generation step, you don't need to run this feature pipeline manually cell-by-cell when working on the full project.

Once you are happy with the rules you have defined in `config/feature_config.yaml` (like adding a new rolling window or a custom clinical score), you can process the entire dataset automatically.

Simply open your terminal and run:

```bash
python -m scripts.02_build_features
```

What happens behind the scenes?

1. The script automatically finds the most recent synthetic data in the `data/synthetic/` folder.

2. It pushes all thousands of patient records through the FeaturePipeline using your YAML configurations.

3. It saves the final, engineered dataset (ready for machine learning or score validation) into a new folder at `data/processed/<timestamp>/`.
